# Spark Architecture & Data Processing Assignment

In [1]:
!pip -q install pyspark pandas

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark=SparkSession.builder.appName("BookstoreAssignment").getOrCreate()
spark

## Step 1: Spark Architecture
- Driver
- Cluster Manager
- Executors
- Local vs Cluster Mode

Spark uses Lazy Evaluation. Transformations build a DAG and execution starts only when an action such as show(), count(), or write() is called.

In [2]:
import pandas as pd

data={
"order_id":[1,2,3,4,5,6,7,8,9,10,11,12],
"customer_name":["Aditi","Rahul","Sneha","Karan","Priya","Aman","Neha","Vikas","Divya","Manoj","Ritu","Suresh"],
"genre":["Fiction","Science","Comics","Fiction","History","Science","Comics","Fiction","Science","History","Fiction","Comics"],
"price":[350,420,180,500,250,390,200,320,450,280,360,190],
"quantity":[2,1,3,2,None,1,4,2,1,3,2,None],
"city":["Hyderabad","Delhi","Mumbai","Pune","Chennai","Hyderabad","Delhi","Mumbai","Pune","Chennai","Delhi","Hyderabad"],
"rating":[4.5,4.2,3.9,4.8,4.1,None,4.0,4.6,4.7,None,4.3,3.8]
}
pd.DataFrame(data).to_csv("bookstore_sales.csv",index=False)
pd.DataFrame(data)

,order_id,customer_name,genre,price,quantity,city,rating
0,1,Aditi,Fiction,350,2.0,Hyderabad,4.5
1,2,Rahul,Science,420,1.0,Delhi,4.2
2,3,Sneha,Comics,180,3.0,Mumbai,3.9
3,4,Karan,Fiction,500,2.0,Pune,4.8
4,5,Priya,History,250,NaN,Chennai,4.1
5,6,Aman,Science,390,1.0,Hyderabad,NaN
6,7,Neha,Comics,200,4.0,Delhi,4.0
7,8,Vikas,Fiction,320,2.0,Mumbai,4.6
8,9,Divya,Science,450,1.0,Pune,4.7
9,10,Manoj,History,280,3.0,Chennai,NaN


In [3]:
schema=StructType([
StructField("order_id",IntegerType(),True),
StructField("customer_name",StringType(),True),
StructField("genre",StringType(),True),
StructField("price",IntegerType(),True),
StructField("quantity",DoubleType(),True),
StructField("city",StringType(),True),
StructField("rating",DoubleType(),True)
])

df=spark.read.csv("bookstore_sales.csv",header=True,schema=schema)
df.printSchema()
df.show()

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- quantity: double (nullable = true)
 |-- city: string (nullable = true)
 |-- rating: double (nullable = true)

+--------+-------------+-------+-----+--------+---------+------+
|order_id|customer_name|  genre|price|quantity|     city|rating|
+--------+-------------+-------+-----+--------+---------+------+
|       1|        Aditi|Fiction|  350|     2.0|Hyderabad|   4.5|
|       2|        Rahul|Science|  420|     1.0|    Delhi|   4.2|
|       3|        Sneha| Comics|  180|     3.0|   Mumbai|   3.9|
|       4|        Karan|Fiction|  500|     2.0|     Pune|   4.8|
|       5|        Priya|History|  250|    NULL|  Chennai|   4.1|
|       6|         Aman|Science|  390|     1.0|Hyderabad|  NULL|
|       7|         Neha| Comics|  200|     4.0|    Delhi|   4.0|
|       8|        Vikas|Fiction|  320|     2.0|   Mumbai|   4.6|
|   

## Lazy Evaluation & DAG

In [4]:
temp=df.filter(col("price")>300).select("customer_name","price")
temp.explain(True)
print("Rows:",temp.count())

== Parsed Logical Plan ==
'Project ['customer_name, 'price]
+- Filter (price#3 > 300)
   +- Relation [order_id#0,customer_name#1,genre#2,price#3,quantity#4,city#5,rating#6] csv

== Analyzed Logical Plan ==
customer_name: string, price: int
Project [customer_name#1, price#3]
+- Filter (price#3 > 300)
   +- Relation [order_id#0,customer_name#1,genre#2,price#3,quantity#4,city#5,rating#6] csv

== Optimized Logical Plan ==
Project [customer_name#1, price#3]
+- Filter (isnotnull(price#3) AND (price#3 > 300))
   +- Relation [order_id#0,customer_name#1,genre#2,price#3,quantity#4,city#5,rating#6] csv

== Physical Plan ==
*(1) Filter (isnotnull(price#3) AND (price#3 > 300))
+- FileScan csv [customer_name#1,price#3] Batched: false, DataFilters: [isnotnull(price#3), (price#3 > 300)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/bookstore_sales.csv], PartitionFilters: [], PushedFilters: [IsNotNull(price), GreaterThan(price,300)], ReadSchema: struct<customer_name:string,price:int>

## Filter and Select

In [5]:
fiction=df.filter(col("genre")=="Fiction").select("customer_name","price","quantity")
fiction.show()

+-------------+-----+--------+
|customer_name|price|quantity|
+-------------+-----+--------+
|        Aditi|  350|     2.0|
|        Karan|  500|     2.0|
|        Vikas|  320|     2.0|
|         Ritu|  360|     2.0|
+-------------+-----+--------+



## Modify DataFrame

In [6]:
df2=df.withColumnRenamed("customer_name","cust_name")
df2=df2.withColumn("price",col("price").cast("double"))
df2=df2.withColumn("total_amount",round(col("price")*col("quantity"),2))
df2.show()

+--------+---------+-------+-----+--------+---------+------+------------+
|order_id|cust_name|  genre|price|quantity|     city|rating|total_amount|
+--------+---------+-------+-----+--------+---------+------+------------+
|       1|    Aditi|Fiction|350.0|     2.0|Hyderabad|   4.5|       700.0|
|       2|    Rahul|Science|420.0|     1.0|    Delhi|   4.2|       420.0|
|       3|    Sneha| Comics|180.0|     3.0|   Mumbai|   3.9|       540.0|
|       4|    Karan|Fiction|500.0|     2.0|     Pune|   4.8|      1000.0|
|       5|    Priya|History|250.0|    NULL|  Chennai|   4.1|        NULL|
|       6|     Aman|Science|390.0|     1.0|Hyderabad|  NULL|       390.0|
|       7|     Neha| Comics|200.0|     4.0|    Delhi|   4.0|       800.0|
|       8|    Vikas|Fiction|320.0|     2.0|   Mumbai|   4.6|       640.0|
|       9|    Divya|Science|450.0|     1.0|     Pune|   4.7|       450.0|
|      10|    Manoj|History|280.0|     3.0|  Chennai|  NULL|       840.0|
|      11|     Ritu|Fiction|360.0|    

## Handle Null Values

In [7]:
df2.select([count(when(col(c).isNull(),c)).alias(c) for c in df2.columns]).show()
clean=df2.na.fill({"rating":4.0,"quantity":1})
clean.show()

+--------+---------+-----+-----+--------+----+------+------------+
|order_id|cust_name|genre|price|quantity|city|rating|total_amount|
+--------+---------+-----+-----+--------+----+------+------------+
|       0|        0|    0|    0|       2|   0|     2|           2|
+--------+---------+-----+-----+--------+----+------+------------+

+--------+---------+-------+-----+--------+---------+------+------------+
|order_id|cust_name|  genre|price|quantity|     city|rating|total_amount|
+--------+---------+-------+-----+--------+---------+------+------------+
|       1|    Aditi|Fiction|350.0|     2.0|Hyderabad|   4.5|       700.0|
|       2|    Rahul|Science|420.0|     1.0|    Delhi|   4.2|       420.0|
|       3|    Sneha| Comics|180.0|     3.0|   Mumbai|   3.9|       540.0|
|       4|    Karan|Fiction|500.0|     2.0|     Pune|   4.8|      1000.0|
|       5|    Priya|History|250.0|     1.0|  Chennai|   4.1|        NULL|
|       6|     Aman|Science|390.0|     1.0|Hyderabad|   4.0|       390.0

## Wide Transformation (Shuffle)

In [8]:
summary=clean.groupBy("genre").sum("total_amount")
summary.show()

+-------+-----------------+
|  genre|sum(total_amount)|
+-------+-----------------+
|Science|           1260.0|
|Fiction|           3060.0|
| Comics|           1340.0|
|History|            840.0|
+-------+-----------------+



## CSV vs Parquet

In [9]:
clean.write.mode("overwrite").option("header",True).csv("output_csv")
clean.write.mode("overwrite").parquet("output_parquet")

pq=spark.read.parquet("output_parquet")
pq.show()

+--------+---------+-------+-----+--------+---------+------+------------+
|order_id|cust_name|  genre|price|quantity|     city|rating|total_amount|
+--------+---------+-------+-----+--------+---------+------+------------+
|       1|    Aditi|Fiction|350.0|     2.0|Hyderabad|   4.5|       700.0|
|       2|    Rahul|Science|420.0|     1.0|    Delhi|   4.2|       420.0|
|       3|    Sneha| Comics|180.0|     3.0|   Mumbai|   3.9|       540.0|
|       4|    Karan|Fiction|500.0|     2.0|     Pune|   4.8|      1000.0|
|       5|    Priya|History|250.0|     1.0|  Chennai|   4.1|        NULL|
|       6|     Aman|Science|390.0|     1.0|Hyderabad|   4.0|       390.0|
|       7|     Neha| Comics|200.0|     4.0|    Delhi|   4.0|       800.0|
|       8|    Vikas|Fiction|320.0|     2.0|   Mumbai|   4.6|       640.0|
|       9|    Divya|Science|450.0|     1.0|     Pune|   4.7|       450.0|
|      10|    Manoj|History|280.0|     3.0|  Chennai|   4.0|       840.0|
|      11|     Ritu|Fiction|360.0|    

## Predicate Pushdown

In [10]:
filtered=pq.filter(col("total_amount")>500)
filtered.explain(True)
filtered.show()

== Parsed Logical Plan ==
'Filter '`>`('total_amount, 500)
+- Relation [order_id#240,cust_name#241,genre#242,price#243,quantity#244,city#245,rating#246,total_amount#247] parquet

== Analyzed Logical Plan ==
order_id: int, cust_name: string, genre: string, price: double, quantity: double, city: string, rating: double, total_amount: double
Filter (total_amount#247 > cast(500 as double))
+- Relation [order_id#240,cust_name#241,genre#242,price#243,quantity#244,city#245,rating#246,total_amount#247] parquet

== Optimized Logical Plan ==
Filter (isnotnull(total_amount#247) AND (total_amount#247 > 500.0))
+- Relation [order_id#240,cust_name#241,genre#242,price#243,quantity#244,city#245,rating#246,total_amount#247] parquet

== Physical Plan ==
*(1) Filter (isnotnull(total_amount#247) AND (total_amount#247 > 500.0))
+- *(1) ColumnarToRow
   +- FileScan parquet [order_id#240,cust_name#241,genre#242,price#243,quantity#244,city#245,rating#246,total_amount#247] Batched: true, DataFilters: [isnotnull

## Pipeline

In [11]:
def pipeline(inp,out):
    d=spark.read.csv(inp,header=True,schema=schema)
    d=d.withColumn("price",col("price").cast("double"))
    d=d.withColumn("total_amount",round(col("price")*coalesce(col("quantity"),lit(1)),2))
    d=d.na.fill({"rating":4.0})
    d=d.filter(col("total_amount")>300)
    d.write.mode("overwrite").parquet(out)
    return d

final_df=pipeline("bookstore_sales.csv","final_output")
final_df.show()
print(final_df.count())

+--------+-------------+-------+-----+--------+---------+------+------------+
|order_id|customer_name|  genre|price|quantity|     city|rating|total_amount|
+--------+-------------+-------+-----+--------+---------+------+------------+
|       1|        Aditi|Fiction|350.0|     2.0|Hyderabad|   4.5|       700.0|
|       2|        Rahul|Science|420.0|     1.0|    Delhi|   4.2|       420.0|
|       3|        Sneha| Comics|180.0|     3.0|   Mumbai|   3.9|       540.0|
|       4|        Karan|Fiction|500.0|     2.0|     Pune|   4.8|      1000.0|
|       6|         Aman|Science|390.0|     1.0|Hyderabad|   4.0|       390.0|
|       7|         Neha| Comics|200.0|     4.0|    Delhi|   4.0|       800.0|
|       8|        Vikas|Fiction|320.0|     2.0|   Mumbai|   4.6|       640.0|
|       9|        Divya|Science|450.0|     1.0|     Pune|   4.7|       450.0|
|      10|        Manoj|History|280.0|     3.0|  Chennai|   4.0|       840.0|
|      11|         Ritu|Fiction|360.0|     2.0|    Delhi|   4.3|